In [ ]:
!uv pip install pandas openpyxl
!uv pip install python-dotenv
!uv pip install langchain_groq
!uv pip install langgraph
!uv pip install langchain_experimental

In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv

# LangChain & LangGraph dependencies
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver

# Visualization
from IPython.display import Image, display

load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY", "")

In [ ]:
data = {
    "Product": ["Laptop", "Monitor", "Keyboard", "Mouse", "Headset"],
    "Region": ["North", "South", "East", "West", "North"],
    "Units_Sold": [120, 85, 300, 450, 190],
    "Unit_Price": [1000, 250, 45, 25, 75],
    "Revenue": [120000, 21250, 13500, 11250, 14250]
}

sample_file_path = os.path.abspath("sample_sales.xlsx")
pd.DataFrame(data).to_excel(sample_file_path, index=False)
print(f"Sample Excel created at: {sample_file_path}")

In [ ]:
# Chat node definig
def chatbot_node(state: MessagesState):
    system_instruction = SystemMessage(
        content=(
            "You are an expert data analyst. You write and execute Python code using `pandas` to analyze Excel files. "
            "When answering questions about the data, write python code that loads the Excel file, calculates the metrics, "
            "and prints the result. Use the output from the tool to formulate your final response."
        )
    )
    messages = [system_instruction] + state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

# 1. Initialize StateGraph using MessagesState
builder = StateGraph(MessagesState)

# 2. Add Nodes
builder.add_node("chatbot", chatbot_node)
builder.add_node("tools", ToolNode(tools))

# 3. Add Edges & Conditional Routing
builder.add_edge(START, "chatbot")
builder.add_conditional_edges("chatbot", tools_condition)
builder.add_edge("tools", "chatbot")

# 4. Compile with Memory Checkpointer
memory = MemorySaver()
graph = builder.compile(checkpointer=memory)

In [ ]:
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print("Graph visualization skipped (requires pygraphviz/mermaid).")

In [ ]:
thread_id = "session_001"
config = {"configurable": {"thread_id": thread_id}}

# Inform the agent about the local file path
init_prompt = f"The user has loaded an Excel file located at: '{sample_file_path}'. Keep this file path in context for subsequent data queries."

response = graph.invoke(
    {"messages": [HumanMessage(content=init_prompt)]},
    config=config
)

print("Agent Initialized:", response["messages"][-1].content)

In [ ]:
def ask_agent(query: str):
    print(f"\n--- User: {query} ---")
    events = graph.stream(
        {"messages": [HumanMessage(content=query)]},
        config=config,
        stream_mode="values"
    )
    
    for event in events:
        last_message = event["messages"][-1]
        
    print(f"Agent:\n{last_message.content}")

# Test Query 1: Aggregation
ask_agent("Which product generated the highest total revenue, and what was the amount of lowest sold product?")

# Test Query 2: Context Memory (Follow-up)
ask_agent("What was the average unit price across all products in that dataset?")